# Building a Research Dataset from Scratch
### ECON 148 — Data Science for Economics

One of the most important — and least glamorous — skills in empirical economics is turning a raw public microdata file into a clean, analysis-ready dataset. This notebook walks through exactly that process for the **CPS Displaced Worker Supplement**, the dataset we will use to study unemployment duration.

By the end of this notebook you will have:

- Understood where the data comes from and what it measures
- Learned how economists construct survival analysis variables from survey data
- Cleaned, recoded, and validated a real microdata extract
- Produced a clean CSV ready for `lifelines`

This workflow — download, inspect, clean, validate, export — is the backbone of virtually every empirical project in economics.

<hr style="border:none; border-top:1px solid #e0e0e0; margin:2em 0;">

## 1. Where does the data come from?

### The Current Population Survey

The **Current Population Survey (CPS)** is the U.S. government's primary source of monthly labor force statistics — it is what produces the official unemployment rate each month. Conducted jointly by the Census Bureau and the Bureau of Labor Statistics, the CPS surveys roughly 60,000 households every month.

Several times a year, the basic monthly survey is expanded with a **supplement** — an additional set of questions on a specific topic. One of these supplements, fielded every two years in January, is the **Displaced Worker Supplement (DWS)**.

### The Displaced Worker Supplement

The DWS asks respondents whether they lost a job in the past three years due to a **layoff, plant closure, or position elimination** — what economists call *involuntary displacement*. For those who did, it collects:

- The industry and occupation of the lost job
- How many weeks they were not working after the job loss
- Whether they found a new job, and if so, at what wage

This is exactly what we need for a survival analysis of unemployment spells.

### IPUMS CPS

The raw CPS microdata is publicly available but awkward to work with — it comes as fixed-width text files with codebooks spanning hundreds of pages, and variable definitions change across years. **IPUMS CPS** (ipums.org), run by the University of Minnesota, solves this by harmonizing variables across decades into consistent codes and providing a point-and-click extract builder that outputs a clean CSV.

### A note on variable names

IPUMS uses its own harmonized variable names which sometimes differ from the raw Census codebook names you might find in papers. The two key variables for survival analysis in this dataset are:

| What we need | IPUMS variable | Description |
|---|---|---|
| Duration | `DWWKSUN` | Weeks not working between end of lost job and start of next job |
| Event | `EMPSTAT` + `DWJOBSINCE` | Current employment status and number of jobs since displacement |

<hr style="border:none; border-top:1px solid #e0e0e0; margin:2em 0;">

## 2. Step-by-step: Downloading from IPUMS

Follow these instructions carefully. The whole process takes about 10–15 minutes the first time.

**Step 1 — Create a free account** at [https://cps.ipums.org/cps/](https://cps.ipums.org/cps/). Click Register, provide your name, institution (UC Berkeley), and email, and agree to the data use agreement. For intended use write something like: *"Classroom instruction on survival analysis of unemployment duration for an undergraduate economics course."*

**Step 2 — Start a new data extract.** Click the green **Get Data** button. You will see an extract builder with a **Data Cart** in the top right.

**Step 3 — Select samples.** Click **Select Samples**. On the samples page:
1. Click the **BASIC MONTHLY** tab (not ASEC)
2. Click the **SUPPLEMENT TOPICS** sub-tab that appears below it
3. Click **DISPLACED WORKER** — a list of biennial January waves appears
4. Check these 11 years: 2022, 2020, 2018, 2016, 2014, 2012, 2010, 2008, 2006, 2004, 2002
5. Click **Submit Sample Selections**

**Step 4 — Select variables.** Add each variable below by searching for it by name in the search box. Click the **+** button to add it to your cart.

| Variable | Description | Where to find it |
|---|---|---|
| `DWSTAT` | Displaced worker status | Search: DWSTAT |
| `DWRESP` | Supplement eligibility/response | Search: DWRESP |
| `DWWKSUN` | Weeks not working after displacement — **our duration variable** | Search: DWWKSUN |
| `DWJOBSINCE` | Number of jobs since losing reference job — **helps construct event** | Search: DWJOBSINCE |
| `DWSUPPWT` | Displaced worker supplement weight | Search: DWSUPPWT |
| `EMPSTAT` | Current employment status — **our event variable** | Search: EMPSTAT |
| `AGE` | Age at survey date | Usually pre-selected |
| `SEX` | Sex | Search: SEX |
| `RACE` | Race | Search: RACE |
| `HISPAN` | Hispanic origin | Search: HISPAN |
| `EDUC` | Educational attainment | Search: EDUC |
| `MARST` | Marital status | Search: MARST |
| `IND1990` | Industry of lost job (harmonized) | Search: IND1990 |

**Step 5 — Choose CSV output.** Click **View Cart**, then **Create Data Extract**. Under **Data Format** select **CSV** (not the default fixed-width). Add a description like `ECON 148 DW supplement 2002-2022` and click **Submit Extract**.

**Step 6 — Download and prepare.** IPUMS will email you in 5–10 minutes. Download the `.csv.gz` file, decompress it, rename it `cps_dw_raw.csv`, and place it in the same folder as this notebook.

**Checklist before continuing:**
- `cps_dw_raw.csv` is in the same folder as this notebook
- The file is uncompressed (not `.gz`)
- You selected all 11 Displaced Worker waves (2002–2022)
- You selected all variables listed above

In [ ]:
#! gunzip cps_00001.csv.gz

# rename cps_00001.csv to cps_dw_raw.csv
#!mv cps_00001.csv cps_dw_raw.csv    


<hr style="border:none; border-top:1px solid #e0e0e0; margin:2em 0;">

## 3. Setup

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker

plt.rcParams.update({
    'figure.dpi': 120,
    'axes.spines.top': False,
    'axes.spines.right': False,
    'axes.grid': True,
    'grid.alpha': 0.3,
})

RAW_PATH = 'cps_dw_raw.csv'
OUT_PATH = 'displacement_survival.csv'

<hr style="border:none; border-top:1px solid #e0e0e0; margin:2em 0;">

## 4. Load and first look

The first thing to do with any new dataset is look at it — literally. Before writing a single line of analysis code, we want to understand the shape of the data, what the columns mean, and what problems might be lurking.

Notice `low_memory=False` — without this, pandas sometimes infers the wrong dtype for columns that mix numbers and missing value codes.

In [ ]:
raw = pd.read_csv(RAW_PATH, low_memory=False)

# IPUMS sometimes delivers column names in lowercase — standardize
raw.columns = raw.columns.str.upper()

print(f'Shape: {raw.shape[0]:,} rows x {raw.shape[1]} columns')
print(f'Columns: {list(raw.columns)}')

In [ ]:
raw.head(10)

The raw extract includes *all* CPS respondents in those January samples — not just displaced workers. Most rows will have missing or **NIU** ("not in universe") values for the DW-specific variables. NIU means the question was simply not asked of that respondent. Our first job is to filter down to the displaced workers only.

In [ ]:
# How many rows per survey year?
print('Total CPS respondents per survey year:')
print(raw['YEAR'].value_counts().sort_index().to_string())

In [ ]:
# Look at the key filter variables before any cleaning
print('DWRESP (supplement eligibility):')
print(raw['DWRESP'].value_counts(dropna=False).to_string())
print()
print('DWSTAT (displaced worker status):')
print(raw['DWSTAT'].value_counts(dropna=False).to_string())

<hr style="border:none; border-top:1px solid #e0e0e0; margin:2em 0;">

## 5. Filter to confirmed displaced workers

The BLS definition of a displaced worker is specific — they must have lost their job due to a **layoff, plant closure, or elimination of their position**, not because they quit, retired, or were fired for cause. `DWSTAT == 1` is IPUMS's harmonized flag for this confirmed displaced status.

In [ ]:
dw = raw[
    raw['DWRESP'].isin([1, 2]) &
    (raw['DWSTAT'] == 1)
].copy()

print(f'All CPS respondents:         {len(raw):>10,}')
print(f'Confirmed displaced workers:  {len(dw):>10,}')
print(f'Share of total sample:        {len(dw)/len(raw):.2%}')

In [ ]:
print('Displaced workers per survey wave:')
print(dw['YEAR'].value_counts().sort_index().to_string())

Notice the spike in 2010 — the Great Recession produced far more displaced workers than any other wave in this series. This is our first piece of real economic signal in the data.

<hr style="border:none; border-top:1px solid #e0e0e0; margin:2em 0;">

## 6. Constructing the survival analysis variables

Survival analysis requires exactly two outcome variables:

1. **`duration`** — how long was the unemployment spell?
2. **`event`** — did the spell end during our observation window? (1 = re-employed, 0 = censored)

### 6a. Duration — `DWWKSUN`

`DWWKSUN` is "number of weeks not working between end of lost job and start of next job." This is exactly the unemployment spell duration we want.

Importantly, `DWWKSUN` is only populated for workers who **found a new job**. For workers still unemployed or out of the labor force at the survey date, it will be zero or missing — those are our censored observations, and we construct their duration differently using `DWJOBSINCE` and `EMPSTAT`.

In [ ]:
# Always inspect a raw variable fully before cleaning
print('DWWKSUN (weeks not working after displacement):')
print(dw['DWWKSUN'].describe().to_string())
print(f'\nMissing / NIU: {dw["DWWKSUN"].isna().sum():,}')
print(f'Zero:          {(dw["DWWKSUN"] == 0).sum():,}')
print()
print('DWJOBSINCE (number of jobs since displacement):')
print(dw['DWJOBSINCE'].value_counts(dropna=False).sort_index().to_string())
print()
print('EMPSTAT (current employment status):')
print(dw['EMPSTAT'].value_counts(dropna=False).sort_index().to_string())

### 6b. Event indicator

We construct the event indicator from two variables:

- **`DWJOBSINCE > 0`** — the worker held at least one job since displacement, meaning re-employment occurred. Combined with `DWWKSUN` this gives us a completed spell duration.
- **`EMPSTAT` codes 10 or 12** — currently employed at survey date (at work, or has job but not at work)

A worker is coded `event = 1` (re-employed) if `DWJOBSINCE >= 1`, meaning they found at least one job after displacement. Everyone else — still unemployed, out of the labor force, or with missing data — is coded `event = 0` (censored).

For censored workers we use `DURUNEMP` (duration of current unemployment spell in weeks) as their duration where available, otherwise we impute from the survey wave's time window.

In [ ]:
# Construct event: 1 = found a job after displacement, 0 = censored
dw['event'] = ((dw['DWJOBSINCE'] >= 1) & dw['DWJOBSINCE'].notna()).astype(int)

print(f'Re-employed (event=1): {dw["event"].sum():,}  ({dw["event"].mean():.1%})')
print(f'Censored    (event=0): {(1-dw["event"]).sum():,}  ({(1-dw["event"]).mean():.1%})')

In [ ]:
# Construct duration:
#   For re-employed workers (event=1): use DWWKSUN (completed spell length)
#   For censored workers  (event=0): use DURUNEMP (weeks searching so far)
#                                     if missing, assign 1 week as lower bound
#
# Cap at 104 weeks (2 years) — older waves coded 99+ as 99, creating
# a pile-up that would distort any model fitted beyond that point.

dw['duration'] = np.where(
    dw['event'] == 1,
    dw['DWWKSUN'],                          # completed spell for re-employed
    dw.get('DURUNEMP', pd.Series(np.nan, index=dw.index))  # current spell for censored
)

# Fill any remaining missing durations with 1 (at least 1 week observed)
dw['duration'] = dw['duration'].fillna(1)

# Drop zeros (ambiguous) and cap at 104 weeks
dw = dw[dw['duration'] > 0].copy()
dw['duration'] = dw['duration'].clip(upper=104).astype(int)

print(f'Rows after duration filter: {len(dw):,}')
print('\nDuration summary (cleaned):')
print(dw['duration'].describe().round(1).to_string())

In [ ]:
fig, ax = plt.subplots(figsize=(9, 4))
ax.hist(dw['duration'], bins=52, color='steelblue', edgecolor='white', linewidth=0.3)
ax.axvline(dw['duration'].median(), color='firebrick', linestyle='--',
           linewidth=1.2, label=f'Median = {dw["duration"].median():.0f} weeks')
ax.set_xlabel('Weeks unemployed after displacement')
ax.set_ylabel('Number of workers')
ax.set_title('Distribution of unemployment spell duration (before accounting for censoring)')
ax.legend()
plt.tight_layout()
plt.show()

In [ ]:
# Sanity check: does re-employment rate vary with macroeconomic conditions?
event_by_year = dw.groupby('YEAR')['event'].agg(['mean', 'count']).round(3)
event_by_year.columns = ['re-employment rate', 'n displaced']

fig, ax = plt.subplots(figsize=(9, 4))
ax.bar(event_by_year.index, event_by_year['re-employment rate'],
       color='steelblue', width=1.2, edgecolor='white')
ax.axvspan(2007.4, 2010.6, alpha=0.12, color='firebrick', label='Great Recession waves')
ax.set_xlabel('Survey year')
ax.set_ylabel('Share re-employed by survey date')
ax.set_title('Re-employment rate by DWS wave\n(should dip in 2008-2010 — sanity check)')
ax.yaxis.set_major_formatter(mticker.PercentFormatter(xmax=1))
ax.legend()
plt.tight_layout()
plt.show()

print(event_by_year.to_string())

The re-employment rate should drop in the 2008 and 2010 waves — exactly when we would expect given the Great Recession. If it does, our coding is working correctly. This kind of sanity check is essential before doing any formal analysis.

<hr style="border:none; border-top:1px solid #e0e0e0; margin:2em 0;">

## 7. What does the data actually look like?

Before building covariates, let's pause and look at the two columns that `lifelines` actually needs — `duration` and `event` — and make sure students understand what each row represents.

In [ ]:
# Show the core survival analysis columns as a clean DataFrame
print('The two things lifelines needs:')
print(f'  duration  — type: {type(dw["duration"]).__name__}, dtype: {dw["duration"].dtype}')
print(f'  event     — type: {type(dw["event"]).__name__},    dtype: {dw["event"].dtype}')
print()
print('Both are just columns in a regular pandas DataFrame.')
print('The KaplanMeierFitter call is simply:')
print('  kmf.fit(df["duration"], event_observed=df["event"])')

In [ ]:
# Show a few rows so the structure is concrete
print('A sample of rows — each one is a displaced worker:')
dw[['YEAR', 'duration', 'event', 'AGE', 'SEX']].head(10)

In [ ]:
# Read a few rows out loud — this is the most important cell
# for building intuition about what the data represents

print('Reading the first 8 rows as plain English:\n')
for i, (_, row) in enumerate(dw.head(8).iterrows()):
    sex = 'woman' if row['SEX'] == 2 else 'man'
    status = 'found a new job' if row['event'] == 1 else 'still searching at survey date (censored)'
    print(f'  Row {i+1}: A {row["AGE"]:.0f}-year-old {sex} displaced in the {row["YEAR"]:.0f} survey '
          f'wave searched for {row["duration"]:.0f} weeks and {status}.')

Each row is one person. The two numbers `lifelines` needs — `duration` and `event` — are all that is required to estimate the survival function. Everything else (age, education, industry) is covariates for the Cox model later.

<hr style="border:none; border-top:1px solid #e0e0e0; margin:2em 0;">

## 8. Constructing covariates

Raw survey codes are rarely analysis-ready. Each section below follows the same pattern: look at raw codes, understand what they mean, recode into clean variables, verify the result.

### 8a. Sex

In [ ]:
# SEX: 1 = male, 2 = female
print('SEX raw codes:')
print(dw['SEX'].value_counts().sort_index().to_string())

dw['female'] = (dw['SEX'] == 2).astype(int)
print(f'\nFemale share: {dw["female"].mean():.1%}')

### 8b. Age

In [ ]:
dw['age'] = dw['AGE']
dw['age_group'] = pd.cut(
    dw['age'],
    bins=[19, 29, 39, 49, 59, 120],
    labels=['20-29', '30-39', '40-49', '50-59', '60+']
)

print('Age group distribution:')
print(dw['age_group'].value_counts().sort_index().to_string())

### 8c. Education

IPUMS harmonizes education into a numeric `EDUC` code. Key values from the codebook:

| EDUC code | Meaning |
|---|---|
| 02–11 | Grades 1–8 |
| 14–30 | Grades 9–11 (some high school) |
| 40 | 12th grade, no diploma |
| 60 | High school diploma or GED |
| 65–73 | Some college or associate's degree |
| 80–110 | Bachelor's degree |
| 111–125 | Advanced degree |

In [ ]:
print('EDUC raw codes:')
print(dw['EDUC'].value_counts().sort_index().to_string())

In [ ]:
def educ_label(code):
    if pd.isna(code): return np.nan
    c = int(code)
    if c < 73:    return 'Less than HS'
    elif c == 73: return 'HS diploma'
    elif c < 111: return 'Some college'
    else:         return 'College plus'

edu_order = ['Less than HS', 'HS diploma', 'Some college', 'College plus']
dw['education'] = dw['EDUC'].apply(educ_label)
dw['education'] = pd.Categorical(dw['education'], categories=edu_order, ordered=True)

print('Education distribution (cleaned):')
print(dw['education'].value_counts().sort_index().to_string())

### 8d. Race and ethnicity

The CPS collects race and Hispanic origin as two separate questions. Following BLS convention, Hispanic identity takes precedence — a respondent who identifies as Hispanic is coded as Hispanic regardless of their racial identification.

In [ ]:
def race_eth(row):
    # HISPAN: 0 = not Hispanic; 901/902 = NIU/missing
    if row['HISPAN'] not in [0, 901, 902]:
        return 'Hispanic'
    r = int(row['RACE'])
    if r == 100:  return 'White non-Hispanic'
    elif r == 200: return 'Black non-Hispanic'
    elif r == 651: return 'Asian non-Hispanic'
    else:          return 'Other/multiracial'

dw['race_eth'] = dw.apply(race_eth, axis=1)

print('Race/ethnicity distribution:')
print(dw['race_eth'].value_counts().to_string())

### 8e. Marital status

In [ ]:
# MARST: 1 = married spouse present — economically meaningful because
# it means the household has a second potential earner, which affects
# job search urgency and reservation wages
dw['married'] = (dw['MARST'] == 1).astype(int)
print(f'Married (spouse present): {dw["married"].mean():.1%}')

### 8f. Industry of lost job

In [ ]:
def industry_label(code):
    if pd.isna(code) or int(code) == 0: return np.nan
    c = int(code)
    if c <= 32:    return 'Agriculture/mining'
    elif c <= 60:  return 'Construction'
    elif c <= 392: return 'Manufacturing'
    elif c <= 472: return 'Trade'
    elif c <= 571: return 'Finance/insurance/RE'
    elif c <= 691: return 'Business/repair services'
    elif c <= 712: return 'Entertainment/hospitality'
    elif c <= 791: return 'Professional services'
    elif c <= 892: return 'Education/health'
    elif c <= 932: return 'Public admin'
    else:          return 'Other'

dw['industry'] = dw['IND1990'].apply(industry_label)

fig, ax = plt.subplots(figsize=(9, 5))
ind_counts = dw['industry'].value_counts().sort_values()
ax.barh(ind_counts.index, ind_counts.values, color='steelblue', edgecolor='white')
ax.set_xlabel('Number of displaced workers')
ax.set_title('Industry of lost job')
plt.tight_layout()
plt.show()

### 8g. Recession indicator

In [ ]:
dw['great_recession'] = dw['YEAR'].isin([2008, 2010]).astype(int)
dw['survey_year'] = dw['YEAR'].astype(int)

print(f'Great Recession waves: {dw["great_recession"].sum():,} obs ({dw["great_recession"].mean():.1%})')

<hr style="border:none; border-top:1px solid #e0e0e0; margin:2em 0;">

## 9. Validation

Before saving anything, we run validation checks. Errors caught here stay out of your results.

In [ ]:
print('Running validation checks...')
print()

assert (dw['duration'] > 0).all(), 'FAIL: zero or negative durations'
print(f'Check 1 passed: all durations > 0  (range: {dw["duration"].min()}-{dw["duration"].max()} weeks)')

assert set(dw['event'].unique()).issubset({0, 1}), 'FAIL: event not binary'
print(f'Check 2 passed: event is binary (0 or 1 only)')

assert dw['duration'].max() <= 104, 'FAIL: duration exceeds 104-week cap'
print(f'Check 3 passed: max duration = {dw["duration"].max()} weeks')

assert dw['age'].between(20, 85).all(), 'FAIL: implausible age values'
print(f'Check 4 passed: age range {dw["age"].min()}-{dw["age"].max()} years')

key_cols = ['duration', 'event', 'female', 'age', 'education']
missing = dw[key_cols].isna().sum()
assert missing.sum() == 0, f'FAIL: missing values\n{missing[missing > 0]}'
print(f'Check 5 passed: no missing values in key columns')

reempl_rate = dw['event'].mean()
assert 0.40 < reempl_rate < 0.95, f'FAIL: re-employment rate {reempl_rate:.1%} implausible'
print(f'Check 6 passed: overall re-employment rate = {reempl_rate:.1%}')

print()
print('All checks passed.')

In [ ]:
# Summary statistics table — Table 1 style
summary = pd.DataFrame({'Statistic': [
    f"{dw['duration'].mean():.1f} weeks",
    f"{dw['duration'].median():.0f} weeks",
    f"{dw['event'].mean():.1%}",
    f"{dw['female'].mean():.1%}",
    f"{dw['age'].mean():.1f} years",
    f"{dw['married'].mean():.1%}",
    f"{(dw['education'] == 'Less than HS').mean():.1%}",
    f"{(dw['education'] == 'HS diploma').mean():.1%}",
    f"{(dw['education'] == 'Some college').mean():.1%}",
    f"{(dw['education'] == 'Bachelors+').mean():.1%}",
    f"{dw['great_recession'].mean():.1%}",
    f"{len(dw):,}",
]}, index=[
    'Mean unemployment duration',
    'Median unemployment duration',
    'Share re-employed (event = 1)',
    'Share female',
    'Mean age',
    'Share married (spouse present)',
    '  Less than high school',
    '  High school diploma',
    '  Some college',
    "  Bachelor's degree or higher",
    'Share in Great Recession waves (2008, 2010)',
    'Total observations',
])

print('Table 1. Summary statistics — CPS Displaced Worker Supplement, 2002-2022')
print('='*60)
print(summary.to_string())

<hr style="border:none; border-top:1px solid #e0e0e0; margin:2em 0;">

## 10. Save the clean dataset

In [ ]:
dw = dw.rename(columns={'DWSUPPWT': 'weight'})

keep = [
    'survey_year',     # DWS wave (2002, 2004, ..., 2022)
    'duration',        # weeks from job loss to re-employment or survey date
    'event',           # 1 = re-employed (event), 0 = censored
    'female',          # 1 = female
    'age',             # continuous age
    'age_group',       # binned age
    'education',       # ordered 4-level categorical
    'race_eth',        # race/ethnicity (Hispanic takes precedence)
    'married',         # 1 = married, spouse present
    'industry',        # broad industry of lost job
    'great_recession', # 1 = displaced in 2008 or 2010 wave
    'weight',          # DW supplement weight
]

out = dw[keep].dropna(subset=['duration', 'event', 'education'])
out.to_csv(OUT_PATH, index=False)

print(f'Saved to: {OUT_PATH}')
print(f'Shape: {len(out):,} rows x {len(out.columns)} columns')
print()
out.head()

<hr style="border:none; border-top:1px solid #e0e0e0; margin:2em 0;">

## 11. What did we just do? A recap

This notebook demonstrates the core data pipeline underlying virtually every empirical economics paper:

**Source → Filter → Construct → Validate → Export**

| Step | What we did | Why it matters |
|---|---|---|
| Source | Downloaded CPS DWS microdata via IPUMS | Reproducible, documented, citable |
| Filter | Kept only confirmed displaced workers (`DWSTAT == 1`) | Sample definition determines what question we can answer |
| Construct `duration` | Used `DWWKSUN` for re-employed, `DURUNEMP` for censored | Duration is the raw material for survival analysis |
| Construct `event` | Used `DWJOBSINCE >= 1` to flag re-employment | Censoring indicator is as important as duration |
| Sanity check | Re-employment rate by year, verified GR dip | Catching coding errors before modeling |
| Show the data | Printed rows as plain English sentences | Built intuition for what each observation means |
| Recode covariates | Education bins, race/ethnicity, industry aggregation | Raw codes are not meaningful — researcher choices matter |
| Validate | Six assertion checks + summary table | Errors caught here stay out of published results |
| Export | Clean CSV with documented column definitions | Reproducible by anyone with the raw IPUMS extract |

Notice how many **researcher judgment calls** went into this — which waves to include, how to cap durations, whether NILF is censored or competing risk, how many education bins. These decisions must be documented clearly. Transparency about data construction is a cornerstone of credible empirical work.